In [0]:
# ══════════════════════════════════════
# 01_FEATURE_ENGINEERING_CLIENTES
# Squad 3 — Etapa de IA/ML
# Segmentação de Consumidores — Lojas Físicas
# ══════════════════════════════════════

# Feature Engineering — Segmentação de Clientes

Objetivo: transformar dados de transação/item (grão fino) em um dataset de 1 linha por cliente
(`cpf_cliente`), com colunas numéricas que descrevem padrões de comportamento de compra —
sem usar RFV, Box-Cox ou K-Means (restrição de escopo do projeto).

**Fontes (Silver, apenas leitura):**
- `physical_vendas_caixa` (grão: transação) — mantida por outro colega da squad
- `physical_itens_venda_caixa` (grão: item de venda)

**Feriados:** não vêm de tabela nenhuma — são gerados deterministicamente com
`gerar_df_feriados_brasil()` (mesma função usada pelo notebook Gold do projeto),
via biblioteca `holidays`, sem tocar no SQL Server.

**Saída:** Delta em `ia/features_clientes` — dataset intermediário, ainda não é Gold,
é o input para a próxima etapa (modelagem/clustering).

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 1. Leitura das fontes Silver

Usa `read_delta()` do `00_utils` — mesma função usada pelo restante do pipeline,
em vez de reescrever `spark.read` na mão.

In [0]:
adls_options = get_adls_options()

df_vendas = read_delta(SILVER_BASE_PATH.rstrip("/") + "/physical_vendas_caixa", adls_options)  # mantida por colega
df_itens  = read_delta(SILVER_ITENS_VENDA_CAIXA_PATH, adls_options)

## 2. Feriados — gerados na hora, sem tabela de origem

Não existe (nunca existiu) uma Silver/Bronze de feriados no projeto. O próprio notebook Gold
gera os feriados deterministicamente via `holidays`. Reaproveitamos a mesma função aqui.

In [0]:
anos_no_dataset = [
    row["ano"] for row in df_vendas.select("ano").distinct().collect()
]

df_feriados_brasil = gerar_df_feriados_brasil(spark, anos_no_dataset)  # colunas: data_feriado, nome_feriado
df_feriados_brasil.show(10, truncate=False)

## 3. Filtro de qualidade — normalização e exclusão de CPF nulo/inválido

Nulo é esperado (cliente não se identifica em toda venda) — essas transações são
excluídas da segmentação, não são "erro". Não existe regra de validação de CPF
documentada em `00_data_quality_rules` — a normalização abaixo é uma decisão desta
etapa, não uma regra oficial já existente no projeto.

In [0]:
df_vendas_validas = (
    df_vendas
    .filter(F.col("cpf_cliente").isNotNull())
    .withColumn(
        "cpf_cliente",
        F.regexp_replace(F.col("cpf_cliente"), r"[^0-9]", "")  # remove pontuação, se houver
    )
    .filter(F.length(F.col("cpf_cliente")) == 11)  # valida 11 dígitos
)

## 4. Filtro de qualidade dos itens

Usa apenas registros já validados pelas flags que a própria Silver disponibiliza.

In [0]:
df_itens_validos = df_itens.filter(
    (F.col("flag_fk_invalido") == False) &
    (F.col("flag_quantidade_invalido") == False) &
    (F.col("flag_preco_invalido") == False) &
    (F.col("flag_valor_inconsistente") == False)
)

## 5. Extração da categoria-base real do produto

`categoria_produto` (Silver) só tem 3 valores possíveis (`perecivel` / `seco` / `NAO_CLASSIFICADO`)
— é uma classificação grosseira, não serve para medir diversidade de produto.

A diversidade real está no `codigo_barras_produto`, que é um slug no formato
`<categoria-base>-<marca>-<peso/volume>-<variante>` (ex.: `queijosfatiadospecas-seara-015kg-minasfrescal`).
O primeiro segmento (antes do primeiro `-`) identifica a categoria-base real do produto —
mesma lógica que o próprio `00_utils` usa para montar `PRODUTO_CATEGORIA_MAP`.

In [0]:
df_itens_validos = df_itens_validos.withColumn(
    "categoria_base_produto",
    F.split(F.col("codigo_barras_produto"), "-").getItem(0)
)


## 6. Join transação + item + feriado

In [0]:
df_base = (
    df_vendas_validas
    .join(df_itens_validos, on="id_transacao", how="inner")
    .join(
        df_feriados_brasil,
        F.to_date(df_vendas_validas.dt_venda) == F.col("data_feriado"),
        how="left",
    )
    .withColumn("venda_em_feriado", F.col("nome_feriado").isNotNull().cast("int"))
)

# Data de referência única para toda a base (não a data real de hoje) —
# evita que clientes cuja última compra é a mais recente da base
# pareçam "inativos" só porque a ingestão parou numa certa data.
data_referencia = df_base.agg(F.max("dt_venda")).collect()[0][0]

## 7. Agregações por cliente

Cada bloco corresponde a um dos 4 grupos de features definidos: comportamento financeiro,
frequência de compra, diversidade de produto, sazonalidade e canal.

### 7.1 Comportamento financeiro

In [0]:
df_financeiro = (
    df_base
    .groupBy("cpf_cliente")
    .agg(
        F.round(F.avg("valor_item_analitico"), 2).alias("ticket_medio"),
        F.round(F.stddev("valor_item_analitico"), 2).alias("desvio_padrao_ticket"),
        F.round(F.sum("valor_item_analitico"), 2).alias("valor_total_gasto"),
    )
)

### 7.2 Frequência de compra

In [0]:
df_datas_por_transacao = (
    df_base
    .select("cpf_cliente", "id_transacao", "dt_venda")
    .distinct()
)

window_cliente = Window.partitionBy("cpf_cliente").orderBy("dt_venda")

df_intervalos = (
    df_datas_por_transacao
    .withColumn("dt_venda_anterior", F.lag("dt_venda").over(window_cliente))
    .withColumn(
        "intervalo_dias",
        F.datediff(F.col("dt_venda"), F.col("dt_venda_anterior"))
    )
)

df_frequencia = (
    df_intervalos
    .groupBy("cpf_cliente")
    .agg(
        F.countDistinct("id_transacao").alias("qtd_transacoes"),
        F.round(F.avg("intervalo_dias"), 1).alias("dias_entre_compras_media"),
        F.max("dt_venda").alias("ultima_compra"),
    )
    .withColumn(
        "dias_desde_ultima_compra",
        F.datediff(F.lit(data_referencia), F.col("ultima_compra"))
    )
    .drop("ultima_compra")
)

### 7.3 Diversidade de produto

Usa `categoria_base_produto` (extraída do slug), não `categoria_produto` (que só tem 3 valores).

In [0]:
df_diversidade = (
    df_base
    .groupBy("cpf_cliente")
    .agg(
        F.countDistinct("categoria_base_produto").alias("qtd_categorias_distintas"),
        F.round(
            F.count("id_item_venda") / F.countDistinct("id_transacao"), 2
        ).alias("qtd_itens_por_transacao_media"),
        F.round(
            100 * F.avg((F.col("categoria_produto") == "perecivel").cast("double")), 1
        ).alias("pct_itens_pereciveis"),
    )
)

Categoria-base dominante (moda) por cliente, já com label encoding.

**Atenção:** label encoding simples (0, 1, 2...) numa variável nominal pode fazer o modelo
de clustering interpretar proximidade artificial entre categorias que não têm ordem real
entre si. Ver observação detalhada no final do notebook.

In [0]:
window_categoria = Window.partitionBy("cpf_cliente").orderBy(F.desc("qtd_compras_categoria"))

df_categoria_dominante = (
    df_base
    .groupBy("cpf_cliente", "categoria_base_produto")
    .agg(F.count("*").alias("qtd_compras_categoria"))
    .withColumn("rank_categoria", F.row_number().over(window_categoria))
    .filter(F.col("rank_categoria") == 1)
    .select("cpf_cliente", F.col("categoria_base_produto").alias("categoria_dominante_nome"))
)

# Label encoding simples (0, 1, 2...) — mantém também o nome em texto
# (categoria_dominante_nome), para permitir reprocessar com one-hot
# ou K-Prototypes na etapa de modelagem sem refazer o pipeline inteiro.
categorias_distintas = (
    df_categoria_dominante.select("categoria_dominante_nome").distinct()
    .withColumn("categoria_dominante", F.row_number().over(Window.orderBy("categoria_dominante_nome")) - 1)
)

df_categoria_dominante = df_categoria_dominante.join(
    categorias_distintas, on="categoria_dominante_nome", how="left"
).select("cpf_cliente", "categoria_dominante", "categoria_dominante_nome")

### 7.4 Sazonalidade e canal

In [0]:
df_feriado_pct = (
    df_datas_por_transacao
    .join(
        df_base.select("cpf_cliente", "id_transacao", "venda_em_feriado").distinct(),
        on=["cpf_cliente", "id_transacao"],
        how="left",
    )
    .groupBy("cpf_cliente")
    .agg(
        F.round(100 * F.avg(F.col("venda_em_feriado").cast("double")), 1).alias("pct_compras_feriado")
    )
)

window_pagamento = Window.partitionBy("cpf_cliente").orderBy(F.desc("qtd_pagamento"))

df_pagamento_dominante = (
    df_vendas_validas
    .groupBy("cpf_cliente", "tipo_pagamento")
    .agg(F.count("*").alias("qtd_pagamento"))
    .withColumn("total_cliente", F.sum("qtd_pagamento").over(Window.partitionBy("cpf_cliente")))
    .withColumn("rank_pagamento", F.row_number().over(window_pagamento))
    .filter(F.col("rank_pagamento") == 1)
    .withColumn(
        "pct_forma_pagamento_dominante",
        F.round(100 * F.col("qtd_pagamento") / F.col("total_cliente"), 1)
    )
    .select("cpf_cliente", "pct_forma_pagamento_dominante")
)

df_lojas_distintas = (
    df_vendas_validas
    .groupBy("cpf_cliente")
    .agg(F.countDistinct("id_loja").alias("qtd_lojas_distintas"))
)


## 8. Junta todos os blocos de feature — grão de 1 linha por cliente

In [0]:
df_features_clientes = (
    df_financeiro
    .join(df_frequencia, on="cpf_cliente", how="inner")
    .join(df_diversidade, on="cpf_cliente", how="inner")
    .join(df_categoria_dominante, on="cpf_cliente", how="inner")
    .join(df_feriado_pct, on="cpf_cliente", how="inner")
    .join(df_pagamento_dominante, on="cpf_cliente", how="inner")
    .join(df_lojas_distintas, on="cpf_cliente", how="inner")
    .withColumn("features_processed_at", F.current_timestamp())
    .withColumn("data_referencia_calculo", F.lit(data_referencia))
    .withColumn("flag_cliente_recorrente", F.col("qtd_transacoes") > 1)
)

print(f"Total de clientes no dataset de features: {df_features_clientes.count()}")
df_features_clientes.printSchema()
df_features_clientes.show(10, truncate=False)


###8.1 Achado de negócio: recorrência de cliente identificado

flag_cliente_recorrente marca clientes com mais de 1 transação no período (qtd_transacoes > 1). Rodando na base completa: apenas ~4% dos clientes com CPF identificado são recorrentes — os outros ~96% fizeram uma única compra identificada no período inteiro.

Isso não é erro de tratamento — é um fato do negócio que muda a estratégia de segmentação (ver .md de definição do modelo, seção "Achado de negócio"). Para a maioria da base, features de frequência/recência (dias_entre_compras_media, desvio_padrao_ticket) ficam nulas ou degeneradas — não discriminam nada.

## 9. Checagem rápida de nulos antes de gravar

`dias_entre_compras_media` será nulo para clientes de 1 única compra — é esperado,
não é erro; decidir na modelagem como tratar.

In [0]:
df_features_clientes.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_features_clientes.columns]
).show(vertical=True)


## 10. Gravação

Dataset intermediário — não é Gold, é input do clustering. Grava em `ia/`, sem
sufixo de versão, seguindo a convenção do restante do projeto. Usa `write_delta()`
do `00_utils` (mesma função usada por Bronze/Silver/Gold), em modo `overwrite`
(reprocessamento completo, não incremental — igual ao padrão da Gold).

In [0]:
IA_FEATURES_CLIENTES_PATH = f"{SQUAD_ROOT_PATH}ia/features_clientes"

write_delta(
    df_features_clientes,
    IA_FEATURES_CLIENTES_PATH,
    adls_options,
    mode="overwrite",
    merge_schema=True,
)

print(f"Features de clientes gravadas em: {IA_FEATURES_CLIENTES_PATH}")


## Observações para a próxima etapa (Modelagem)

1. **`categoria_dominante`** usa label encoding simples (0,1,2...), decisão tomada nesta etapa.
   Atenção: por ser variável nominal sem ordem natural, algoritmos baseados em distância
   (Hierárquico, GMM) podem interpretar erroneamente proximidade entre códigos. O nome original
   (`categoria_dominante_nome`) foi mantido no dataset para permitir reprocessar com one-hot
   encoding ou trocar para K-Prototypes sem refazer o pipeline inteiro.
2. **`dias_entre_compras_media`** é nulo para clientes com apenas 1 transação — decidir se esses
   clientes formam um segmento próprio ("compra única") ou se são imputados/excluídos antes
   do clustering.
3. **`categoria_base_produto`** foi extraída por split simples do `codigo_barras_produto`
   (segmento antes do primeiro `-`) — mesma lógica de origem do `PRODUTO_CATEGORIA_MAP`
   do `00_utils`, mas sem passar pelo dicionário perecível/seco. Não tem a mesma revisão
   humana recomendada para esse dicionário — tratar como aproximação, não fonte oficial.
4. Todas as colunas numéricas precisam de **padronização** (`StandardScaler` ou similar) antes
   do clustering — escalas muito diferentes hoje (ex.: `valor_total_gasto` em reais vs
   `qtd_categorias_distintas` em unidades).
5. Não existe regra de validação formal de `cpf_cliente` no `00_data_quality_rules` — a
   normalização usada aqui (remover pontuação, exigir 11 dígitos) é uma decisão local desta
   etapa, ainda não validada contra uma amostra real de valores.